# 🔬 Crack-Distill: KD Boost Research (Notebook 09)

This notebook systematically tests **4 new KD techniques** not yet evaluated, each designed to push beyond the current best (Mask mAP50 = 0.5569, Affinity KD).

| # | Technique | Core Idea | Expected Benefit |
|:---:|:---|:---|:---|
| A | **Focal Mask-KL** | Weight KL by `(1-p_student)^γ` — focus on uncertain crack boundary pixels | Better sub-pixel contour learning |
| B | **Jensen-Shannon Divergence** | Symmetric KL: `JSD = 0.5·KL(S‖T) + 0.5·KL(T‖S)` | Smoother gradients, less mode collapse |
| C | **Affinity + Dilated Combined** | `03_dilated`'s focused mask weighting + `04_affinity`'s gradient matching in one loss | Best of both proven winners |
| D | **Cosine-Annealed Temperature** | Start at τ=6 (max soft) → anneal to τ=2 (sharpening) over 150 epochs | Better boundary sharpening at late epochs |

### Shared Config:
* **Student**: YOLOv11n-seg (2.84M params, 10.2 GFLOPs)
* **Teacher**: SAM 2 Large (pre-computed soft logits from `distill_datasetforme`)
* **Epochs**: 150 (full)
* **Seed**: 42
* **Precision**: FP32 (no AMP)
* **Base W**: 0.9612


In [ ]:
# ── Environment & Directory Initialization ──
!mkdir -p configs utils distillation scripts checkpoints data/datasets data/teacher_logits_box runs results
!pip install -q ultralytics albumentations pycocotools thop pyyaml pandas tqdm opencv-python Pillow


In [ ]:
%%writefile configs/config.yaml
# ============================================================
# Crack-Distill — Master Config
# ============================================================
# YOU ONLY NEED TO CHANGE:
#   1. data.datasets[0].path  → your crack500_yolo folder
#   2. student.backbone       → yolo11n-seg (demo) or yolo11s-seg (paper)
#   3. train.epochs           → 10 (demo) or 100 (real)
# Everything else leave as is.
# ============================================================

project:
  name: crack-distill
  seed: 42
  output_dir: runs/

# ----------------------------------------------------------
# TASK
# To switch: change type ONLY — everything else adapts.
# Options: instance_seg | semantic_seg | detection
# ----------------------------------------------------------
task:
  type: instance_seg
  num_classes: 1
  class_names:
    - crack

# ----------------------------------------------------------
# DATASET  ← change path here
# ----------------------------------------------------------
data:
  root: data/datasets/
  train_split: 0.8
  val_split: 0.1
  test_split: 0.1
  image_size: 512
  batch_size: 16
  num_workers: 4

  datasets:
    - name: combined
      path: data/datasets/combined_yolo
      format: yolo

# ----------------------------------------------------------
# TEACHER (SAM 2)  ← no changes needed
# ----------------------------------------------------------
teacher:
  model: sam2
  checkpoint: checkpoints/sam2_hiera_large.pt
  config: configs/sam2.1/sam2.1_hiera_l.yaml  # sam2 package config
  device: cuda
  prompt_type: box
  save_logits: True
  logits_dir: data/teacher_logits/
  batch_size: 4

# ----------------------------------------------------------
# STUDENT
# ----------------------------------------------------------
student:
  backbone: yolo11n-seg   # primary student backbone (YOLOv11)
  pretrained: True
  device: "0,1"
  imgsz: 512

# ----------------------------------------------------------
# KD LOSSES  ← no changes needed
# L = L_task + α·L_mask_kd + β·L_feature + γ·L_boundary
# ----------------------------------------------------------
distillation:
  enabled: True
  temperature: 3.7769      # tuned via composite OOD-aware Optuna search

  progressive:
    enabled: True
    stage1_pct: 0.30
    stage2_pct: 0.70

  losses:
    task:
      weight: 1.0
    mask_kd:
      enabled: True
      weight: 0.9612       # tuned KL divergence mask weight
    feature:
      enabled: True
      weight: 1.8658       # tuned intermediate feature alignment weight
      layers: [2, 5, 8]
    boundary:
      enabled: True
      weight: 0.8055       # reduced to prevent over-memorizing crop boundary artifacts

# ----------------------------------------------------------
# TRAINING  ← change epochs here
# ----------------------------------------------------------
train:
  epochs: 150    # 10 for demo, 150 for real training
  lr: 0.001
  lr_scheduler: cosine
  warmup_epochs: 3
  optimizer: AdamW
  weight_decay: 0.0005
  amp: False     # disabled FP16 AMP to prevent loss NaN / FP16 overflow on thin crack datasets (e.g. DeepCrack)
  grad_clip: 10.0
  save_every: 10
  patience: 20

# ----------------------------------------------------------
# EVALUATION  ← no changes needed
# ----------------------------------------------------------
eval:
  primary_metric: mAP50-seg
  metrics:
    - mAP50-seg
    - mAP50-95-seg
    - dice
    - boundary_iou
    - fps
    - latency_ms
    - params_M
    - gflops

# ----------------------------------------------------------
# PAPER EXPERIMENTS  ← no changes needed
# ----------------------------------------------------------
experiments:
  - name: baseline_finetune
    distillation.enabled: False
  - name: pseudo_labels
    distillation.losses.mask_kd.enabled: False
    distillation.losses.feature.enabled: False
    distillation.losses.boundary.enabled: False
  - name: full_kd_box
    distillation.enabled: True
    teacher.logits_dir: data/teacher_logits_box/
  - name: full_kd_centroid
    distillation.enabled: True
    teacher.logits_dir: data/teacher_logits_centroid/
  - name: low_data_5pct
    data.train_fraction: 0.05
  - name: low_data_10pct
    data.train_fraction: 0.10
  - name: low_data_25pct
    data.train_fraction: 0.25
  - name: low_data_50pct
    data.train_fraction: 0.50


In [ ]:
%%writefile utils/__init__.py
# utils package


In [ ]:
%%writefile utils/config_loader.py
"""Config loader — converts YAML to a dot-access object."""

import yaml
from pathlib import Path


class ConfigNode:
    """Dot-access config object. cfg.data.batch_size just works."""

    def __init__(self, d: dict):
        for k, v in d.items():
            if isinstance(v, dict):
                setattr(self, k, ConfigNode(v))
            elif isinstance(v, list):
                setattr(self, k, [
                    ConfigNode(i) if isinstance(i, dict) else i for i in v
                ])
            else:
                setattr(self, k, v)

    def get(self, key, default=None):
        return getattr(self, key, default)

    def keys(self):
        return self.__dict__.keys()

    def values(self):
        return self.__dict__.values()

    def items(self):
        return self.__dict__.items()

    def __getitem__(self, key):
        if hasattr(self, key):
            return getattr(self, key)
        raise KeyError(key)

    def __setitem__(self, key, value):
        setattr(self, key, value)

    def __contains__(self, key):
        return hasattr(self, key)

    def __repr__(self):
        return f"ConfigNode({self.__dict__})"

    def __iter__(self):
        return iter(self.__dict__.keys())

    def dict(self):
        result = {}
        for k, v in self.__dict__.items():
            if isinstance(v, ConfigNode):
                result[k] = v.dict()
            elif isinstance(v, list):
                result[k] = [i.dict() if isinstance(i, ConfigNode) else i for i in v]
            else:
                result[k] = v
        return result


def load_config(path: str) -> ConfigNode:
    """Load YAML config and return dot-access ConfigNode."""
    with open(path) as f:
        raw = yaml.safe_load(f)
    return ConfigNode(raw)


def override_config(cfg: ConfigNode, overrides: dict) -> ConfigNode:
    """
    Apply flat-key overrides to a config.
    e.g. override_config(cfg, {"distillation.enabled": False})
    """
    raw = cfg.dict()
    for key_path, value in overrides.items():
        parts = key_path.split(".")
        node = raw
        for p in parts[:-1]:
            node = node.setdefault(p, {})
        node[parts[-1]] = value
    return ConfigNode(raw)


In [ ]:
%%writefile distillation/__init__.py
# distillation package


In [ ]:
%%writefile distillation/kd_trainer.py
"""
KD Boost Research Trainer — 4 New Techniques
==============================================
Technique selector: set KD_VARIANT env var BEFORE calling train_with_kd().
VARIANT is read fresh on every compute_kd_loss() call (not at import time),
so switching variants in the same kernel works without restarting.
"""
import os, math
import torch
import torch.nn.functional as F
import numpy as np
from pathlib import Path
from ultralytics import YOLO
from ultralytics.models.yolo.segment import SegmentationTrainer
from ultralytics.utils.loss import v8SegmentationLoss
from ultralytics.utils import RANK

# ── Global logit cache ──────────────────────────────────────────
_logit_cache = {}

def load_logit(stem, logit_dir):
    if stem not in _logit_cache:
        p = Path(logit_dir) / f"{stem}_logits.npy"
        _logit_cache[stem] = torch.from_numpy(np.load(str(p))).float() if p.exists() else None
    return _logit_cache[stem]

# ── Technique A: Focal Mask-KL ──────────────────────────────────
def focal_kl_loss(s_logits, t_logits, tau=3.7769, gamma=2.0):
    """KL weighted by (1 - p_student)^gamma — focuses gradient on uncertain boundary pixels."""
    s = torch.sigmoid(s_logits / tau)
    t = torch.sigmoid(t_logits / tau)
    # Per-pixel focal weight: hard for the student = high weight
    focal_w = (1.0 - s).detach() ** gamma
    # Binary KL per pixel
    eps = 1e-7
    kl = t * (torch.log(t + eps) - torch.log(s + eps)) + \
         (1-t) * (torch.log(1-t + eps) - torch.log(1-s + eps))
    loss = (focal_w * kl).mean()
    return tau ** 2 * loss

# ── Technique B: Jensen-Shannon Divergence ───────────────────────
def jsd_loss(s_logits, t_logits, tau=3.7769):
    """Symmetric KL — JSD = 0.5*KL(S||M) + 0.5*KL(T||M) where M = 0.5*(S+T)."""
    s = torch.sigmoid(s_logits / tau)
    t = torch.sigmoid(t_logits / tau)
    m = 0.5 * (s + t)
    eps = 1e-7
    def kl_bern(p, q):
        return p * (torch.log(p + eps) - torch.log(q + eps)) + \
               (1-p) * (torch.log(1-p + eps) - torch.log(1-q + eps))
    loss = 0.5 * kl_bern(s, m).mean() + 0.5 * kl_bern(t, m).mean()
    return tau ** 2 * loss

# ── Technique C: Affinity + Dilated Combined ─────────────────────
def combined_affinity_dilated_loss(s_logits, t_logits, tau=3.7769, affinity_w=0.5):
    """Foreground-dilated Mask-KL (03_dilated) + Spatial Affinity (04_affinity) combined."""
    s = torch.sigmoid(s_logits / tau)
    t = torch.sigmoid(t_logits / tau)
    eps = 1e-7

    # Part 1: Foreground-dilated KL (from 03_dilated)
    # Build region weight mask: crack core=1.0, 8px band=0.5, background=0.05
    fg = (t > 0.5).float()
    dilated = F.max_pool2d(fg, kernel_size=9, stride=1, padding=4)
    w_map = torch.where(fg > 0.5, torch.ones_like(fg), 
            torch.where(dilated > 0.5, 0.5 * torch.ones_like(fg), 0.05 * torch.ones_like(fg)))
    kl_pix = t * (torch.log(t + eps) - torch.log(s + eps)) + \
             (1-t) * (torch.log(1-t + eps) - torch.log(1-s + eps))
    dilated_loss = tau ** 2 * (w_map * kl_pix).sum() / (w_map.sum() + eps)

    # Part 2: 4-directional spatial affinity (from 04_affinity)
    dx_s = s[:, :, :, 1:] - s[:, :, :, :-1]
    dx_t = t[:, :, :, 1:] - t[:, :, :, :-1]
    dy_s = s[:, :, 1:, :] - s[:, :, :-1, :]
    dy_t = t[:, :, 1:, :] - t[:, :, :-1, :]
    affinity_loss = F.mse_loss(dx_s, dx_t) + F.mse_loss(dy_s, dy_t)

    return dilated_loss + affinity_w * affinity_loss

# ── Technique D: Cosine-Annealed Temperature ─────────────────────
_current_epoch = [0]
_total_epochs = [150]

def cosine_tau(epoch, total, tau_max=6.0, tau_min=2.0):
    """Anneal temperature from tau_max (soft) to tau_min (sharp) over training."""
    progress = epoch / max(total - 1, 1)
    return tau_min + 0.5 * (tau_max - tau_min) * (1 + math.cos(math.pi * progress))

def cosine_tau_kl_loss(s_logits, t_logits, epoch=0, total=150):
    """Standard Mask-KL but with epoch-annealed temperature."""
    tau = cosine_tau(epoch, total)
    s = torch.sigmoid(s_logits / tau)
    t = torch.sigmoid(t_logits / tau)
    eps = 1e-7
    kl = t * (torch.log(t + eps) - torch.log(s + eps)) + \
         (1-t) * (torch.log(1-t + eps) - torch.log(1-s + eps))
    return tau ** 2 * kl.mean()

# ── KD Loss Dispatcher ───────────────────────────────────────────
# NOTE: reads os.environ FRESH on every call — no module-level global.
# This means changing KD_VARIANT in the same kernel always takes effect.
def compute_kd_loss(s_logits, t_logits, epoch=0, total=150):
    variant = os.environ.get("KD_VARIANT", "focal_kl")  # read fresh every call
    if variant == "focal_kl":
        return focal_kl_loss(s_logits, t_logits)
    elif variant == "jsd":
        return jsd_loss(s_logits, t_logits)
    elif variant == "combined":
        return combined_affinity_dilated_loss(s_logits, t_logits)
    elif variant == "cosine_tau":
        return cosine_tau_kl_loss(s_logits, t_logits, epoch, total)
    else:
        raise ValueError(f"Unknown KD_VARIANT: {variant}")

# ── Patched Segmentation Loss ────────────────────────────────────
class KDSegLoss(v8SegmentationLoss):
    def __init__(self, model, logit_dir, kd_weight=0.9612):
        super().__init__(model)
        self.logit_dir = logit_dir
        self.kd_weight = kd_weight
        self._epoch = 0
        self._total = 150

    def __call__(self, preds, batch):
        base_loss, base_items = super().__call__(preds, batch)
        try:
            device = base_loss.device
            proto = preds[1][1] if isinstance(preds[1], (list, tuple)) else preds[1]
            B = proto.shape[0]
            kd_losses = []
            for i in range(B):
                stem = Path(batch["im_file"][i]).stem
                t_logits = load_logit(stem, self.logit_dir)
                if t_logits is None:
                    continue
                t_logits = t_logits.to(device)
                s_proto = proto[i:i+1]  # (1, C, H, W)
                s_logit = s_proto.mean(dim=1, keepdim=True)  # (1, 1, H, W)
                H, W = t_logits.shape[-2:]
                if s_logit.shape[-2:] != (H, W):
                    s_logit = F.interpolate(s_logit, (H, W), mode="bilinear", align_corners=False)
                if t_logits.dim() == 2:
                    t_logits = t_logits.unsqueeze(0).unsqueeze(0)
                kd_losses.append(compute_kd_loss(s_logit, t_logits, self._epoch, self._total))
            if kd_losses:
                kd_loss = torch.stack(kd_losses).mean()
                base_loss = base_loss + self.kd_weight * kd_loss
        except Exception as e:
            pass  # never crash training
        return base_loss, base_items

# ── KD Trainer ───────────────────────────────────────────────────
class KDSegmentationTrainer(SegmentationTrainer):
    def __init__(self, logit_dir, kd_weight=0.9612, **kwargs):
        super().__init__(**kwargs)
        self.logit_dir = logit_dir
        self.kd_weight = kd_weight

    def get_model(self, cfg=None, weights=None, verbose=True):
        model = super().get_model(cfg=cfg, weights=weights, verbose=verbose)
        return model

    def _setup_train(self, *args, **kwargs):
        super()._setup_train(*args, **kwargs)
        self.loss_fn = KDSegLoss(self.model, self.logit_dir, self.kd_weight)
        self.loss_fn._total = self.epochs

    def optimizer_step(self):
        if hasattr(self, "loss_fn"):
            self.loss_fn._epoch = self.epoch
        super().optimizer_step()


def train_with_kd(data_yaml, logit_dir, run_name, kd_weight=0.9612, epochs=150, imgsz=512, seed=42):
    """Main entry point. Prints active variant to log for verification."""
    active = os.environ.get("KD_VARIANT", "focal_kl")
    print(f"[KD] ✅ Active variant: {active}")
    print(f"[KD]    Run name      : {run_name}")
    print(f"[KD]    KD weight     : {kd_weight}")
    print(f"[KD]    Epochs        : {epochs}")
    trainer = KDSegmentationTrainer(
        logit_dir=logit_dir,
        kd_weight=kd_weight,
        overrides={
            "model": "yolo11n-seg.pt",
            "data": data_yaml,
            "epochs": epochs,
            "imgsz": imgsz,
            "batch": 16,
            "seed": seed,
            "amp": False,
            "name": run_name,
            "project": "runs",
            "verbose": True,
        }
    )
    trainer.train()
    return trainer


In [ ]:
%%writefile scripts/convert_crack500.py
#!/usr/bin/env python3
"""
Crack500 → YOLO seg format converter
=====================================
Crack500 structure:
  crack500/
  ├── traincrop/   ← 00001.jpg + 00001.png (binary mask, same stem)
  ├── valcrop/
  ├── testcrop/
  ├── train.txt    ← list of image filenames (optional)
  ├── val.txt
  └── test.txt

Output (YOLO seg format, ready for ultralytics):
  crack500_yolo/
  ├── images/
  │   ├── train/
  │   ├── val/
  │   └── test/
  ├── labels/
  │   ├── train/
  │   ├── val/
  │   └── test/
  └── dataset.yaml

Each .txt label: one line per connected crack instance
  0 x1 y1 x2 y2 ... (normalized polygon, class 0 = crack)

Usage:
  python scripts/convert_crack500.py \
      --src ~/distill/data/datasets/crack500 \
      --dst ~/distill/data/datasets/crack500_yolo
"""

import os
import cv2
import numpy as np
import argparse
import shutil
from pathlib import Path
from tqdm import tqdm


CLASS_ID = 0        # single class: crack
MIN_AREA = 50       # minimum pixel area to keep an instance
MIN_POINTS = 6      # minimum polygon points (3 coordinate pairs)


def binary_mask_to_yolo_instances(mask_path: str, img_w: int, img_h: int) -> list[str]:
    """
    Read binary PNG mask → split into instances via connectedComponents
    → convert each to normalized YOLO seg polygon string.

    Returns list of label lines (one per instance).
    """
    mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
    if mask is None:
        return []

    # Threshold (Crack500 masks are 0/255)
    binary = (mask > 127).astype(np.uint8)

    # Separate touching cracks into individual instances
    num_labels, labels_map = cv2.connectedComponents(binary)

    label_lines = []
    for label_id in range(1, num_labels):      # 0 = background
        instance = (labels_map == label_id).astype(np.uint8)

        if instance.sum() < MIN_AREA:
            continue

        # Find contours for this instance
        contours, _ = cv2.findContours(
            instance, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE
        )

        for contour in contours:
            if len(contour) < MIN_POINTS // 2:
                continue

            # Flatten and normalize to [0, 1]
            pts = contour.squeeze()
            if pts.ndim == 1:
                pts = pts.reshape(1, 2)

            # Simplify contour slightly to reduce file size
            epsilon = 0.002 * cv2.arcLength(contour, True)
            simplified = cv2.approxPolyDP(contour, epsilon, True).squeeze()
            if simplified.ndim == 1:
                simplified = simplified.reshape(1, 2)
            if len(simplified) < 3:
                simplified = pts

            norm = []
            for x, y in simplified:
                norm.append(x / img_w)
                norm.append(y / img_h)

            if len(norm) < MIN_POINTS:
                continue

            coords_str = " ".join(f"{v:.6f}" for v in norm)
            label_lines.append(f"{CLASS_ID} {coords_str}")

    return label_lines


def process_split(src_dir: Path, dst_img_dir: Path, dst_lbl_dir: Path, split_name: str):
    """Process one split (train/val/test)."""

    # Crack500 stores images+masks together in traincrop/valcrop/testcrop
    crop_dir = src_dir / f"{split_name}crop"
    if not crop_dir.exists():
        # Try alternate names
        for candidate in [src_dir / split_name, src_dir / f"{split_name}data"]:
            if candidate.exists():
                crop_dir = candidate
                break
        else:
            print(f"  [WARNING] Could not find directory for split '{split_name}', skipping.")
            return 0

    dst_img_dir.mkdir(parents=True, exist_ok=True)
    dst_lbl_dir.mkdir(parents=True, exist_ok=True)

    # Find all images (jpg/jpeg/png that are NOT masks)
    all_files = sorted(crop_dir.iterdir())
    # Crack500: image = .jpg, mask = same stem + .png
    image_files = [f for f in all_files if f.suffix.lower() in ('.jpg', '.jpeg')
                   and ':Zone.Identifier' not in f.name]

    if not image_files:
        # Some versions store as .png images too — distinguish by paired files
        png_files = [f for f in all_files if f.suffix.lower() == '.png'
                     and ':Zone.Identifier' not in f.name]
        # If .jpg exists for a stem → .png is mask. If no .jpg → .png is image.
        jpg_stems = {f.stem for f in all_files if f.suffix.lower() in ('.jpg', '.jpeg')}
        image_files = [f for f in png_files if f.stem not in jpg_stems]

    converted = 0
    skipped   = 0

    for img_path in tqdm(image_files, desc=f"  {split_name}", leave=False):
        stem = img_path.stem

        # Find corresponding mask (.png with same stem)
        mask_path = crop_dir / f"{stem}.png"
        if not mask_path.exists():
            # Try .bmp
            mask_path = crop_dir / f"{stem}.bmp"
        if not mask_path.exists():
            skipped += 1
            continue

        # Read image to get dimensions
        img = cv2.imread(str(img_path))
        if img is None:
            skipped += 1
            continue
        h, w = img.shape[:2]

        # Convert mask to YOLO seg labels
        label_lines = binary_mask_to_yolo_instances(str(mask_path), w, h)

        # Copy image
        dst_img_path = dst_img_dir / img_path.name
        shutil.copy2(img_path, dst_img_path)

        # Write label file (even if empty — YOLO needs it)
        dst_lbl_path = dst_lbl_dir / f"{stem}.txt"
        with open(dst_lbl_path, "w") as f:
            f.write("\n".join(label_lines))

        converted += 1

    print(f"  {split_name}: {converted} images converted, {skipped} skipped")
    return converted


def write_dataset_yaml(dst: Path, num_train: int, num_val: int, num_test: int):
    """Write ultralytics-compatible dataset.yaml."""
    yaml_content = f"""# Crack500 — YOLO seg format
# Auto-generated by convert_crack500.py

path: {dst.resolve()}
train: images/train
val:   images/val
test:  images/test

nc: 1
names:
  0: crack

# Stats
# train: ~{num_train} images
# val:   ~{num_val} images
# test:  ~{num_test} images
"""
    with open(dst / "dataset.yaml", "w") as f:
        f.write(yaml_content)
    print(f"\n  dataset.yaml written to {dst / 'dataset.yaml'}")


def verify_conversion(dst: Path):
    """Quick sanity check on converted dataset."""
    print("\n[Verify] Checking converted dataset...")
    issues = 0
    for split in ["train", "val", "test"]:
        img_dir = dst / "images" / split
        lbl_dir = dst / "labels" / split
        if not img_dir.exists():
            continue

        imgs = list(img_dir.glob("*.jpg")) + list(img_dir.glob("*.png"))
        lbls = list(lbl_dir.glob("*.txt"))

        # Check counts match
        if len(imgs) != len(lbls):
            print(f"  [!] {split}: {len(imgs)} images vs {len(lbls)} labels — mismatch!")
            issues += 1
        else:
            print(f"  {split}: {len(imgs)} images ✓")

        # Check a few labels are non-empty
        non_empty = sum(1 for l in lbls if l.stat().st_size > 0)
        empty     = len(lbls) - non_empty
        print(f"    labels with cracks: {non_empty} | empty (no crack): {empty}")

        if non_empty == 0:
            print(f"  [!] {split}: ALL labels are empty — check mask paths!")
            issues += 1

    if issues == 0:
        print("\n  ✓ Dataset looks good!")
    else:
        print(f"\n  ✗ {issues} issue(s) found — check output above.")

    return issues == 0


def main():
    parser = argparse.ArgumentParser(description="Convert Crack500 to YOLO seg format")
    parser.add_argument(
        "--src",
        type=str,
        required=True,
        help="Path to crack500 root dir (contains traincrop/, valcrop/, testcrop/)"
    )
    parser.add_argument(
        "--dst",
        type=str,
        default=None,
        help="Output directory (default: <src>_yolo)"
    )
    parser.add_argument(
        "--verify",
        action="store_true",
        default=True,
        help="Run sanity check after conversion"
    )
    args = parser.parse_args()

    src = Path(args.src).expanduser().resolve()
    dst = Path(args.dst).expanduser().resolve() if args.dst else src.parent / f"{src.name}_yolo"

    print(f"[Convert] Source: {src}")
    print(f"[Convert] Output: {dst}")
    print()

    if not src.exists():
        print(f"ERROR: Source directory not found: {src}")
        return

    counts = {}
    for split in ["train", "val", "test"]:
        n = process_split(
            src_dir    = src,
            dst_img_dir= dst / "images" / split,
            dst_lbl_dir= dst / "labels" / split,
            split_name = split,
        )
        counts[split] = n

    write_dataset_yaml(dst, counts["train"], counts["val"], counts["test"])

    if args.verify:
        verify_conversion(dst)

    print(f"\n[Done] Converted dataset at: {dst}")
    print(f"\nNext step — test YOLO11 loads it:")
    print(f"  from ultralytics import YOLO")
    print(f"  model = YOLO('yolo11n-seg.pt')")
    print(f"  model.train(data='{dst}/dataset.yaml', epochs=1, imgsz=512)")


if __name__ == "__main__":
    main()
# (appended — nothing, file is complete)


In [ ]:
%%writefile scripts/convert_crack500_uncropped.py
#!/usr/bin/env python3
"""
Crack500 Uncropped Test/Val → YOLO seg format converter
======================================================
Converts the original uncropped validation and test sets of Crack500.
Handles EXIF orientation for images by rotating the corresponding masks.

Source directories:
  data/datasets/crack500/valdata/   ← contains {stem}.jpg and {stem}_mask.png
  data/datasets/crack500/testdata/  ← contains {stem}.jpg and {stem}_mask.png

Output (YOLO seg format):
  data/datasets/crack500_uncropped_yolo/
  ├── images/
  │   ├── val/
  │   └── test/
  ├── labels/
  │   ├── val/
  │   └── test/
  └── dataset.yaml
"""

import os
import cv2
import numpy as np
import argparse
import shutil
from pathlib import Path
from tqdm import tqdm
from PIL import Image


CLASS_ID = 0        # single class: crack
MIN_AREA = 50       # minimum pixel area to keep an instance
MIN_POINTS = 6      # minimum polygon points (3 coordinate pairs)


def get_exif_rotation(img_path: Path):
    """Retrieve EXIF orientation tag from image."""
    try:
        with Image.open(img_path) as im:
            exif = im.getexif()
            if exif:
                return exif.get(274)  # 274 is the Orientation tag
    except Exception:
        pass
    return None


def rotate_mask_to_match_image(mask: np.ndarray, exif_orientation: int) -> np.ndarray:
    """Rotate mask array to match image rotation applied by cv2.imread based on EXIF."""
    if exif_orientation == 6:
        return cv2.rotate(mask, cv2.ROTATE_90_CLOCKWISE)
    elif exif_orientation == 8:
        return cv2.rotate(mask, cv2.ROTATE_90_COUNTERCLOCKWISE)
    elif exif_orientation == 3:
        return cv2.rotate(mask, cv2.ROTATE_180)
    return mask


def binary_mask_to_yolo_instances(mask_path: str, img_w: int, img_h: int, exif_orientation: int = None) -> list[str]:
    """
    Read binary PNG mask → rotate based on EXIF → split into instances via connectedComponents
    → convert each to normalized YOLO seg polygon string.
    """
    mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
    if mask is None:
        return []

    if exif_orientation:
        mask = rotate_mask_to_match_image(mask, exif_orientation)

    # Threshold (Crack500 masks are binary 0/255)
    binary = (mask > 127).astype(np.uint8)

    # Separate touching cracks into individual instances
    num_labels, labels_map = cv2.connectedComponents(binary)

    label_lines = []
    for label_id in range(1, num_labels):      # 0 = background
        instance = (labels_map == label_id).astype(np.uint8)

        if instance.sum() < MIN_AREA:
            continue

        # Find contours for this instance
        contours, _ = cv2.findContours(
            instance, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE
        )

        for contour in contours:
            if len(contour) < MIN_POINTS // 2:
                continue

            # Flatten and normalize to [0, 1]
            pts = contour.squeeze()
            if pts.ndim == 1:
                pts = pts.reshape(1, 2)

            # Simplify contour slightly to reduce file size
            epsilon = 0.002 * cv2.arcLength(contour, True)
            simplified = cv2.approxPolyDP(contour, epsilon, True).squeeze()
            if simplified.ndim == 1:
                simplified = simplified.reshape(1, 2)
            if len(simplified) < 3:
                simplified = pts

            norm = []
            for x, y in simplified:
                norm.append(x / img_w)
                norm.append(y / img_h)

            if len(norm) < MIN_POINTS:
                continue

            coords_str = " ".join(f"{v:.6f}" for v in norm)
            label_lines.append(f"{CLASS_ID} {coords_str}")

    return label_lines


def process_split(src_dir: Path, dst_dir: Path, split_name: str):
    """Process uncropped val or test split."""
    split_dir = src_dir / f"{split_name}data"
    if not split_dir.exists():
        print(f"  [Warning] Directory {split_dir} does not exist, skipping split {split_name}.")
        return 0

    dst_img_dir = dst_dir / "images" / split_name
    dst_lbl_dir = dst_dir / "labels" / split_name

    dst_img_dir.mkdir(parents=True, exist_ok=True)
    dst_lbl_dir.mkdir(parents=True, exist_ok=True)

    # Find all image files (jpg/jpeg/png that do not contain '_mask')
    all_files = sorted(split_dir.iterdir())
    image_files = [
        f for f in all_files 
        if f.suffix.lower() in ('.jpg', '.jpeg', '.png')
        and '_mask' not in f.name.lower()
        and ':Zone.Identifier' not in f.name
    ]

    converted = 0
    skipped = 0

    for img_path in tqdm(image_files, desc=f"  {split_name}", leave=False):
        stem = img_path.stem

        # Find mask (stem + "_mask.png")
        mask_path = split_dir / f"{stem}_mask.png"
        if not mask_path.exists():
            skipped += 1
            continue

        # Read image to get dimensions (matches how cv2.imread auto-rotates it based on EXIF)
        img = cv2.imread(str(img_path))
        if img is None:
            skipped += 1
            continue
        h, w = img.shape[:2]

        # Get EXIF rotation from image
        exif_orientation = get_exif_rotation(img_path)

        # Convert mask to YOLO seg labels (rotating it to match)
        label_lines = binary_mask_to_yolo_instances(str(mask_path), w, h, exif_orientation)

        # Copy image
        dst_img_path = dst_img_dir / img_path.name
        shutil.copy2(img_path, dst_img_path)

        # Write label file (even if empty — YOLO needs it)
        dst_lbl_path = dst_lbl_dir / f"{stem}.txt"
        with open(dst_lbl_path, "w") as f:
            f.write("\n".join(label_lines))

        converted += 1

    print(f"  {split_name}: {converted} images converted, {skipped} skipped")
    return converted


def main():
    parser = argparse.ArgumentParser(description="Convert Crack500 Uncropped splits to YOLO seg format")
    parser.add_argument(
        "--src",
        type=str,
        default="data/datasets/crack500",
        help="Path to crack500 root dir"
    )
    parser.add_argument(
        "--dst",
        type=str,
        default="data/datasets/crack500_uncropped_yolo",
        help="Output directory"
    )
    args = parser.parse_args()

    src = Path(args.src).expanduser().resolve()
    dst = Path(args.dst).expanduser().resolve()

    print(f"[Convert] Source: {src}")
    print(f"[Convert] Output: {dst}")
    print()

    if not src.exists():
        print(f"ERROR: Source directory not found: {src}")
        return

    if dst.exists():
        print(f"[Warning] Output directory exists, clearing: {dst}")
        shutil.rmtree(dst)

    dst.mkdir(parents=True, exist_ok=True)

    counts = {}
    for split in ["val", "test"]:
        n = process_split(src, dst, split)
        counts[split] = n

    # Write dataset_uncropped.yaml
    yaml_content = f"""# Crack500 Uncropped — YOLO seg format
# Auto-generated by convert_crack500_uncropped.py

path: {dst.resolve()}
train: images/val
val:   images/val
test:  images/test

nc: 1
names:
  0: crack

# Stats
# val:   ~{counts.get('val', 0)} images (uncropped)
# test:  ~{counts.get('test', 0)} images (uncropped)
"""
    with open(dst / "dataset.yaml", "w") as f:
        f.write(yaml_content)
    print(f"\n  dataset.yaml written to {dst / 'dataset.yaml'}")
    print(f"[Done] Converted uncropped splits successfully.")


if __name__ == "__main__":
    main()


In [ ]:
%%writefile scripts/generate_teacher_logits.py
#!/usr/bin/env python3
"""
Generate SAM 2 teacher logits for crack500 training images.
Run ONCE before training. Saves .npy logit files to data/teacher_logits/

Usage (from ~/distill):
  python scripts/generate_teacher_logits.py           # full dataset
  python scripts/generate_teacher_logits.py --resume  # skip already done
  python scripts/generate_teacher_logits.py --max 50  # only first N images
"""

import argparse
import sys
import os
import cv2
import numpy as np
import torch
from pathlib import Path

# ── project root on path ─────────────────────────────────────
ROOT = Path(__file__).parent.parent.resolve()
sys.path.insert(0, str(ROOT))

# ── fixed paths (relative to ~/distill) ─────────────────────
DATASET_DIR = ROOT / "data/datasets/crack500"
LOGITS_DIR  = ROOT / "data/teacher_logits"
SAM2_CKPT   = ROOT / "checkpoints/sam2_hiera_large.pt"
SAM2_CFG    = "configs/sam2.1/sam2.1_hiera_l.yaml"


def mask_to_bbox(mask: np.ndarray):
    rows = np.any(mask, axis=1)
    cols = np.any(mask, axis=0)
    if not rows.any():
        return None
    r1, r2 = np.where(rows)[0][[0, -1]]
    c1, c2 = np.where(cols)[0][[0, -1]]
    return np.array([c1, r1, c2, r2], dtype=np.float32)


def mask_to_centroid(mask: np.ndarray):
    M = cv2.moments(mask)
    if M["m00"] != 0:
        cX = int(M["m10"] / M["m00"])
        cY = int(M["m01"] / M["m00"])
        h, w = mask.shape[:2]
        if 0 <= cX < w and 0 <= cY < h and mask[cY, cX] > 0:
            return np.array([[cX, cY]], dtype=np.float32)
    # Fallback to maximum of distance transform (guaranteed to be inside mask)
    dist_transform = cv2.distanceTransform(mask, cv2.DIST_L2, 5)
    _, _, _, max_loc = cv2.minMaxLoc(dist_transform)
    cX, cY = max_loc
    return np.array([[cX, cY]], dtype=np.float32)


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--resume", action="store_true",
                        help="Skip images that already have logits")
    parser.add_argument("--max", type=int, default=None,
                        help="Max images to process (for testing)")
    parser.add_argument("--img-dir", type=str, default=None,
                        help="Path to directory of training images")
    parser.add_argument("--mask-dir", type=str, default=None,
                        help="Path to directory of training masks")
    parser.add_argument("--prefix", type=str, default="crack500_",
                        help="Prefix to use for saved logits files")
    parser.add_argument("--prompt-type", type=str, default="box_centroid",
                        choices=["box", "box_centroid"],
                        help="Prompt type to use (box or box_centroid)")
    parser.add_argument("--logits-dir", type=str, default=None,
                        help="Override output logits directory")
    parser.add_argument("--dataset", type=str, default=None,
                        help="Path to YOLO dataset or raw dataset directory")
    args = parser.parse_args()

    if args.dataset and not args.img_dir:
        ds_path = Path(args.dataset).expanduser().resolve()
        if (ds_path / "images/train").exists():
            args.img_dir = str(ds_path / "images/train")
            args.mask_dir = str(ds_path / "masks/train") if (ds_path / "masks/train").exists() else str(ds_path / "images/train")
        elif ds_path.exists():
            args.img_dir = str(ds_path)
            args.mask_dir = str(ds_path)

    if args.logits_dir:
        logits_dir = Path(args.logits_dir).expanduser().resolve()
    else:
        logits_dir = ROOT / "data/teacher_logits"
        
    orig_logits_dir = logits_dir
    # On Kaggle, redirect to /tmp/ to avoid exceeding 20GB disk limit
    if "/kaggle/" in str(logits_dir):
        logits_dir = Path("/tmp") / logits_dir.name
        logits_dir.mkdir(parents=True, exist_ok=True)
        try:
            if os.path.lexists(orig_logits_dir):
                if os.path.islink(orig_logits_dir):
                    os.unlink(orig_logits_dir)
                elif orig_logits_dir.is_dir() and not list(orig_logits_dir.glob("*.npy")):
                    orig_logits_dir.rmdir()
            if not orig_logits_dir.exists():
                orig_logits_dir.parent.mkdir(parents=True, exist_ok=True)
                os.symlink(logits_dir, orig_logits_dir)
                print(f"[Logits] Created symlink: {orig_logits_dir} -> {logits_dir}")
        except Exception as e:
            print(f"[Logits Warning] Could not symlink {orig_logits_dir} -> {logits_dir}: {e}")
        
    logits_dir.mkdir(parents=True, exist_ok=True)

    if args.img_dir:
        # Single custom dataset mode
        datasets = [{
            "name": "Custom",
            "img_dir": Path(args.img_dir).expanduser().resolve(),
            "mask_dir": Path(args.mask_dir).expanduser().resolve() if args.mask_dir else Path(args.img_dir).expanduser().resolve(),
            "prefix": args.prefix
        }]
    else:
        # Multi-dataset auto-mode
        datasets = [
            {
                "name": "Crack500",
                "img_dir": ROOT / "data/datasets/crack500/traincrop",
                "mask_dir": ROOT / "data/datasets/crack500/traincrop",
                "prefix": "crack500_"
            },
            {
                "name": "DeepCrack",
                "img_dir": ROOT / "data/datasets/deepcrack/train_img",
                "mask_dir": ROOT / "data/datasets/deepcrack/train_lab",
                "prefix": "deepcrack_"
            }
        ]

    # Filter out non-existent datasets
    active_datasets = []
    for d in datasets:
        if d["img_dir"].exists():
            active_datasets.append(d)
        else:
            if args.img_dir:
                print(f"ERROR: Image directory not found: {d['img_dir']}")
                return
            else:
                print(f"Skipping {d['name']} logits generation: directory {d['img_dir']} does not exist.")

    if not active_datasets:
        print("No active datasets to process. Exiting.")
        return

    # ── Load SAM 2 ───────────────────────────────────────────
    print(f"Loading SAM 2 from {SAM2_CKPT}...")
    from sam2.build_sam import build_sam2
    from sam2.sam2_image_predictor import SAM2ImagePredictor

    model     = build_sam2(SAM2_CFG, str(SAM2_CKPT), device="cuda")
    predictor = SAM2ImagePredictor(model)
    print("SAM 2 loaded ✓\n")

    for d in active_datasets:
        img_dir = d["img_dir"]
        mask_dir = d["mask_dir"]
        prefix = d["prefix"]

        # ── Process images ───────────────────────────────────────
        # Check for jpg/jpeg files first to avoid loading PNG masks as images in Crack500 traincrop
        images = sorted([
            f for f in img_dir.iterdir()
            if f.suffix.lower() in ('.jpg', '.jpeg')
            and ':Zone.Identifier' not in f.name
        ])
        if not images:
            images = sorted([
                f for f in img_dir.iterdir()
                if f.suffix.lower() in ('.jpg', '.jpeg', '.png')
                and ':Zone.Identifier' not in f.name
            ])
        if args.max:
            images = images[:args.max]

        print(f"\n>>> Processing {d['name']} ({len(images)} images) <<<")
        print(f"  Images: {img_dir}")
        print(f"  Masks:  {mask_dir}")
        print(f"  Prefix: {prefix}\n")

        generated = 0
        skipped   = 0
        failed    = 0

        for img_path in images:
            image_id   = f"{prefix}{img_path.stem}"
            logit_file = logits_dir / f"{image_id}_logits.npy"

            # Skip if already done
            if args.resume and logit_file.exists():
                skipped += 1
                continue

            # Load image
            image = cv2.imread(str(img_path))
            if image is None:
                failed += 1
                continue
            image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

            # Load instance masks from binary PNG/BMP file or YOLO label .txt file
            instance_masks = []

            mask_path = mask_dir / f"{img_path.stem}.png"
            if not mask_path.exists():
                mask_path = mask_dir / f"{img_path.stem}.bmp"
            
            # Check raw dataset fallback paths (e.g. train_lab / traincrop)
            if not mask_path.exists():
                for alt_name in ["train_lab", "traincrop", "masks"]:
                    alt_path = img_dir.parent.parent / alt_name / f"{img_path.stem}.png"
                    if alt_path.exists():
                        mask_path = alt_path
                        break

            # Check YOLO txt label fallback
            label_txt_path = img_dir.parent.parent / "labels" / img_dir.name / f"{img_path.stem}.txt"

            if mask_path.exists():
                binary = cv2.imread(str(mask_path), cv2.IMREAD_GRAYSCALE)
                if binary is not None:
                    binary = (binary > 127).astype(np.uint8)
                    num_labels, labels_map = cv2.connectedComponents(binary)
                    for label_id in range(1, num_labels):
                        inst = (labels_map == label_id).astype(np.uint8)
                        if inst.sum() >= 50:
                            instance_masks.append(inst)
            elif label_txt_path.exists():
                img_h, img_w = image.shape[:2]
                with open(label_txt_path, "r") as lf:
                    lines = lf.readlines()
                for line in lines:
                    parts = line.strip().split()
                    if len(parts) < 7:  # class + at least 3 (x,y) pairs
                        continue
                    try:
                        coords = [float(x) for x in parts[1:]]
                        pts = np.array(coords).reshape(-1, 2)
                        pts[:, 0] *= img_w
                        pts[:, 1] *= img_h
                        inst = np.zeros((img_h, img_w), dtype=np.uint8)
                        cv2.fillPoly(inst, [pts.astype(np.int32)], 1)
                        if inst.sum() >= 50:
                            instance_masks.append(inst)
                    except Exception:
                        pass

            if not instance_masks:
                failed += 1
                continue

            all_logits = []
            predictor.set_image(image)

            # Capture encoder features
            features = predictor._features
            image_embed = features["image_embed"].cpu().half().numpy()
            feat0 = features["high_res_feats"][0].cpu().half().numpy()
            feat1 = features["high_res_feats"][1].cpu().half().numpy()

            for instance in instance_masks:
                box = mask_to_bbox(instance)
                if box is None:
                    continue

                if args.prompt_type == "box_centroid":
                    centroid = mask_to_centroid(instance)
                    point_labels = np.array([1], dtype=np.int32)
                    with torch.no_grad():
                        _, _, logits = predictor.predict(
                            point_coords=centroid,
                            point_labels=point_labels,
                            box=box,
                            multimask_output=False,
                        )
                else:
                    with torch.no_grad():
                        _, _, logits = predictor.predict(
                            box=box,
                            multimask_output=False,
                        )
                all_logits.append(logits[0].astype(np.float32))  # (256, 256)

            if all_logits:
                np.save(str(logit_file), np.stack(all_logits, axis=0))
                
                # Save encoder features in a shared features directory to save disk space
                features_dir = logits_dir.parent / "teacher_features"
                features_dir.mkdir(parents=True, exist_ok=True)
                feat_file = features_dir / f"{image_id}_features.npz"
                if not feat_file.exists():
                    np.savez_compressed(
                        str(feat_file),
                        image_embed=image_embed,
                        feat1=feat1
                    )
                generated += 1
            else:
                failed += 1

            total = generated + skipped + failed
            if total % 50 == 0 or total == len(images):
                print(f"  {total}/{len(images)} | "
                      f"generated={generated} skipped={skipped} failed={failed}")

        print(f"\n  Finished {d['name']}: generated={generated}, skipped={skipped}, failed={failed}")

    print("\n[Done] All active datasets processed successfully.")
    print("Next step:")
    print("  python scripts/run_experiments.py --exp full_kd")


if __name__ == "__main__":
    main()


In [ ]:
# ── Step 1: Link Kaggle Inputs (Dataset & Teacher Logits) ──
import os, shutil
from pathlib import Path

input_dir = Path("/kaggle/input/distill_datasetforme")
if not input_dir.exists():
    input_dir = Path("/kaggle/input")

datasets_dir = Path("data/datasets")
datasets_dir.mkdir(parents=True, exist_ok=True)
checkpoints_dir = Path("checkpoints")
checkpoints_dir.mkdir(parents=True, exist_ok=True)

# 1. Link Crack500 raw images
found_dataset = False
for root, dirs, files in os.walk(str(input_dir)):
    root_path = Path(root)
    if "traincrop" in dirs:
        dest = datasets_dir / "crack500"
        if os.path.lexists(dest):
            os.unlink(dest) if os.path.islink(dest) else shutil.rmtree(dest)
        os.symlink(root_path, dest)
        print(f"[Dataset] Linked Crack500: {root_path} -> {dest}")
        found_dataset = True
        break

# 2. Link precomputed teacher logits
found_logits = False
for root, dirs, files in os.walk(str(input_dir)):
    root_p = Path(root)
    for target_name in ["teacher_logits_box", "teacher_logits"]:
        if target_name in dirs:
            src_f = root_p / target_name
            dst_f = Path("data/teacher_logits_box")
            if os.path.lexists(dst_f):
                os.unlink(dst_f) if os.path.islink(dst_f) else shutil.rmtree(dst_f)
            os.symlink(src_f, dst_f)
            print(f"[Logits] Linked precomputed logits: {src_f} -> {dst_f}")
            found_logits = True
            break
    if found_logits:
        break


In [ ]:
# ── Step 2: Convert Datasets & Verify Teacher Logits ──
# 1. Convert Cropped Crack500
!python scripts/convert_crack500.py --src data/datasets/crack500 --dst data/datasets/crack500_yolo

# 2. Convert Uncropped Crack500 for OOD Evaluation
if Path("data/datasets/crack500/valdata").exists():
    !python scripts/convert_crack500_uncropped.py --src data/datasets/crack500 --dst data/datasets/crack500_uncropped_yolo

# 3. Verify Logits
logits_dir = Path("data/teacher_logits_box")
logits_count = len(list(logits_dir.glob("*_logits.npy"))) if logits_dir.exists() else 0
print(f"[Verification] Found {logits_count} teacher logit files in {logits_dir}")

if logits_count == 0:
    print("=== Pre-computed logits not found in input; Generating SAM 2 Logits on GPU ===")
    ckpt_file = checkpoints_dir / "sam2_hiera_large.pt"
    if not ckpt_file.exists():
        !wget -q https://dl.fbaipublicfiles.com/segment_anything_2/092824/sam2.1_hiera_large.pt -O checkpoints/sam2_hiera_large.pt
    !pip install -q SAM-2 || pip install -q git+https://github.com/facebookresearch/segment-anything-2.git
    !python scripts/generate_teacher_logits.py --prompt-type box --logits-dir data/teacher_logits_box --dataset data/datasets/crack500_yolo

logits_count = len(list(logits_dir.glob("*_logits.npy")))
assert logits_count > 0, f"[FATAL ERROR] 0 logit files found in {logits_dir}. KD training cannot proceed without teacher supervision!"
print(f"[Verification Passed] Ready to train with {logits_count} real SAM 2 teacher logits from {logits_dir}!")


In [ ]:
# ── Step 3: Run All 4 KD Boost Variants ──
# Uncomment exactly ONE block to run that variant.
# Each runs 150 epochs. Expected Kaggle runtime: ~2.5–3.0 hrs per variant.
# Attach: distill_datasetforme dataset.

import os, sys, json
sys.path.insert(0, ".")
from pathlib import Path
from ultralytics import YOLO

LOGIT_DIR  = "data/teacher_logits_box"
DATA_YAML  = "data/datasets/crack500_yolo/dataset.yaml"
KD_WEIGHT  = 0.9612
EPOCHS     = 150
SEED       = 42

# ════════════════════════════════════════════════════════════════
# VARIANT A — Focal Mask-KL (γ=2.0)
# Focuses KL gradient on uncertain boundary pixels.
# ════════════════════════════════════════════════════════════════
os.environ["KD_VARIANT"] = "focal_kl"
from distillation.kd_trainer import train_with_kd
trainer_A = train_with_kd(
    data_yaml=DATA_YAML,
    logit_dir=LOGIT_DIR,
    run_name="09A_focal_kl_g2_T3.7769_W0.9612_seed42_150ep",
    kd_weight=KD_WEIGHT,
    epochs=EPOCHS,
    seed=SEED,
)

# ════════════════════════════════════════════════════════════════
# VARIANT B — Jensen-Shannon Divergence
# Symmetric KL — smoother gradients, no mode collapse risk.
# ════════════════════════════════════════════════════════════════
# os.environ["KD_VARIANT"] = "jsd"
# from distillation.kd_trainer import train_with_kd  # reimport reloads VARIANT
# trainer_B = train_with_kd(
#     data_yaml=DATA_YAML, logit_dir=LOGIT_DIR,
#     run_name="09B_jsd_T3.7769_W0.9612_seed42_150ep",
#     kd_weight=KD_WEIGHT, epochs=EPOCHS, seed=SEED,
# )

# ════════════════════════════════════════════════════════════════
# VARIANT C — Affinity + Dilated Combined
# Best of 03_dilated (focused region) + 04_affinity (topology).
# ════════════════════════════════════════════════════════════════
# os.environ["KD_VARIANT"] = "combined"
# from distillation.kd_trainer import train_with_kd
# trainer_C = train_with_kd(
#     data_yaml=DATA_YAML, logit_dir=LOGIT_DIR,
#     run_name="09C_combined_affinity_dilated_W0.9612_seed42_150ep",
#     kd_weight=KD_WEIGHT, epochs=EPOCHS, seed=SEED,
# )

# ════════════════════════════════════════════════════════════════
# VARIANT D — Cosine-Annealed Temperature (τ: 6→2 over 150 ep)
# Starts very soft (max dark knowledge), sharpens at late epochs.
# ════════════════════════════════════════════════════════════════
# os.environ["KD_VARIANT"] = "cosine_tau"
# from distillation.kd_trainer import train_with_kd
# trainer_D = train_with_kd(
#     data_yaml=DATA_YAML, logit_dir=LOGIT_DIR,
#     run_name="09D_cosine_tau_6to2_W0.9612_seed42_150ep",
#     kd_weight=KD_WEIGHT, epochs=EPOCHS, seed=SEED,
# )


In [ ]:
# ── Step 4: Evaluate Best Checkpoint & Compare vs. Existing Baselines ──
import glob, json, os
from pathlib import Path
from ultralytics import YOLO

# NOTE on baselines:
#   No-KD (EXP-10) 0.5400 is the TRUE no-KD in-domain baseline (from nb_1_runned.ipynb).
#   All others (01_seed42 onward) are KD runs — comparisons among them are KD vs. KD.
#   The no-KD OOD number is still MISSING (pending a rerun of the no-KD ckpt through nb07).
KNOWN_BASELINES = {
    "No-KD baseline (EXP-10)": {"Mask mAP50": 0.5400, "Box mAP50": 0.5970},
    "01_seed42 [plain Mask-KL KD]": {"Mask mAP50": 0.5424, "Box mAP50": 0.5976},
    "03_dilated [fgd-dilated KL KD]": {"Mask mAP50": 0.5387, "Box mAP50": 0.5819},
    "04_affinity [affinity KD]": {"Mask mAP50": 0.5569, "Box mAP50": 0.5973},
    "05_multiscale [512 logit KD]": {"Mask mAP50": 0.5485, "Box mAP50": 0.6001},
    "06_layerkd [neck CWD KD]": {"Mask mAP50": 0.5538, "Box mAP50": 0.5947},
}

DATA_YAML = "data/datasets/crack500_yolo/dataset.yaml"
OOD_YAML  = "data/datasets/crack500_uncropped_yolo/dataset.yaml"

# Find this run's best checkpoint
ckpts = sorted(Path("runs").rglob("best.pt"), key=lambda p: p.stat().st_mtime, reverse=True)
if not ckpts:
    raise RuntimeError("No best.pt found. Run training first.")

best_ckpt = ckpts[0]
run_name = best_ckpt.parent.parent.name
print(f"Evaluating: {run_name}")
print(f"Checkpoint: {best_ckpt}")

model = YOLO(str(best_ckpt))

# In-domain eval
res_id = model.val(data=DATA_YAML, split="val", verbose=False)
mask_map50 = float(res_id.seg.map50)
box_map50  = float(res_id.box.map50)
mask_map95 = float(res_id.seg.map)

# OOD eval
import os
ood_mask_map50 = None
if os.path.exists(OOD_YAML):
    res_ood = model.val(data=OOD_YAML, split="val", verbose=False)
    ood_mask_map50 = float(res_ood.seg.map50)

new_result = {
    "run_name": run_name,
    "Mask mAP50": mask_map50,
    "Mask mAP50-95": mask_map95,
    "Box mAP50": box_map50,
    "OOD Mask mAP50": ood_mask_map50,
}

# Save result
os.makedirs("results", exist_ok=True)
out_path = f"results/{run_name}.json"
with open(out_path, "w") as f:
    json.dump(new_result, f, indent=2)

print()
print("=" * 65)
print("📊 HEAD-TO-HEAD COMPARISON vs. EXISTING BASELINES")
print("=" * 65)
print(f"  {'Model':<30} {'Mask mAP50':>12} {'Box mAP50':>12}")
print("-" * 65)
for name, vals in KNOWN_BASELINES.items():
    print(f"  {name:<30} {vals['Mask mAP50']:>12.4f} {vals['Box mAP50']:>12.4f}")
print("-" * 65)
print(f"  {run_name:<30} {mask_map50:>12.4f} {box_map50:>12.4f}  ← THIS RUN")
print("=" * 65)

delta = mask_map50 - 0.5400
vs_affinity = mask_map50 - 0.5569
print(f"  vs. No-KD in-domain baseline (0.5400, EXP-10): {delta:+.4f} ({delta/0.5400*100:+.1f}%)")
print(f"  vs. Best KD so far (04_affinity, 0.5569):       {vs_affinity:+.4f} ({vs_affinity/0.5569*100:+.1f}%)")
if ood_mask_map50 is not None:
    ood_vs_kd = ood_mask_map50 - 0.0848
    print(f"  OOD vs. 01_seed42 plain-KL (0.0848) [KD vs KD]: {ood_vs_kd:+.4f} ({ood_vs_kd/0.0848*100:+.1f}%)")
    print(f"  ⚠️  True no-KD OOD baseline still missing — run no-KD ckpt through nb07 first.")
print()
print(f"Full result saved to: {out_path}")
